# Proyecto de simulación matematica 

### Paso 1: Importar Librerías Esenciales

Necesitamos 4 librerías principales:
- **Polars**: Procesamiento rápido de datos (mejor que Pandas)
- **Pandas**: Compatibilidad con PVLib
- **Math**: Funciones trigonométricas (sin, cos, pi, etc.)
- **PVLib**: Cálculos solares profesionales (posición del sol, radiación)


In [ ]:
# ============================================================================
# IMPORTACIÓN DE LIBRERÍAS NECESARIAS
# ============================================================================
# Estas librerías son fundamentales para el procesamiento de datos y cálculos
# solares del proyecto

import polars as pl       # Procesamiento de datos de gran volumen (muy eficiente)
import pandas as pd       # Manipulación de datos auxiliar (compatibilidad con PVLib)
import math              # Funciones matemáticas avanzadas (pi, exp, cos, sqrt, etc.)
import pvlib             # Cálculos solares: posición del sol, irradiancia, etc.


### Paso 2: Definir Constantes Geográficas y Físicas

#### Ubicación (Morelia, México)
- **Latitud**: 19.7° N (norte del ecuador)
- **Longitud**: -101.18° O (oeste de Greenwich)  
- **Altitud**: 1920 metros sobre el nivel del mar

#### Parámetros Ópticos
| Componente | Valor | Significado |
|-----------|-------|-------------|
| **Lente** | 49 cm ⌀ | Área = 0.188 m² |
| **Transmitancia** | 85% | Cuánta luz pasa |
| **Absortancia** | 95% | Cuánta radiación absorbe |

#### Agua a Calentar
- **Entrada**: 20°C → **Salida**: 94°C
- **Diferencia**: 74°C
- **Energía requerida**: ~1.4 MJ por lote


In [ ]:
# ============================================================================
# 1. CONFIGURACIÓN INICIAL Y CONSTANTES DEL PROYECTO
# ============================================================================
# Ubicación: Morelia, Michoacán, México
# Este bloque define todas las constantes necesarias para los cálculos solares

# --- 1.1. PARÁMETROS GEOGRÁFICOS Y DE UBICACIÓN ---
phi = 19.7008                   # Latitud de Morelia (grados norte del ecuador)
longitud = -101.1844            # Longitud de Morelia (grados oeste del meridiano de Greenwich)
h_alt = 1.92                    # Altitud en kilómetros para modelo de Hottel (transmitancia atmosférica)
altitud_m = 1920                # Altitud en metros para cálculos de PVLib
zona_horaria = 'America/Mexico_City' # Zona horaria para cálculos solares precisos
beta = 0                        # Inclinación del colector respecto a la horizontal (0° = horizontal)

# --- 1.2. CONSTANTE SOLAR Y MODELO ATMOSFÉRICO (HOTTEL) ---
# La radiación solar extraterrestre varía con la distancia Tierra-Sol según la estación
# El modelo de Hottel calcula la transmitancia atmosférica considerando diferentes altitudes
Gsc = 1367                      # Constante Solar (W/m²) - energía solar fuera de la atmósfera

# Coeficientes de transmitancia atmosférica según modelo de Hottel
# Estos valores varían con la altitud y modelan cómo la atmósfera filtra la radiación solar
Ao = 0.4237 - 0.00821 * (6 - h_alt)**2 # Coeficiente de radiación directa (componente sin dispersión)
A1 = 0.5055 + 0.00595 * (6.5 - h_alt)**2 # Coeficiente de extinción (absorción y dispersión en atmósfera)
K = 0.2711 + 0.01858 * (2.5 - h_alt)**2 # Coeficiente de reflexión del terreno (albedo)

# --- 1.3. PROPIEDADES ÓPTICAS DE LA LENTE CONCENTRADORA ---
# La lente es el componente clave que concentra la radiación solar
radio_lente = 0.245             # Radio de la lente en metros (diámetro de 49 cm)
Aa = math.pi * (radio_lente**2) # Área de apertura de la lente (aproximadamente 0.188 m²)
radio_foco = 0.05               # Radio del foco (zona donde se concentra la energía)

# --- 1.4. PROPIEDADES DE TRANSMITANCIA Y ABSORTANCIA ---
# Estos valores determinan cuánta energía solar realmente llega al fluido
tau_lente = 0.85                # Transmitancia de la lente (85% de la luz la atraviesa)
tau_vidrio = 0.90               # Transmitancia del vidrio de protección (90% de transmisión)
alpha_cobre = 0.95              # Absortancia del tubo de cobre (absorbe 95% de la radiación)
epsilon_p = 0.95                # Emisividad del tubo de cobre (irradia mucho calor)

# --- 1.5. PROPIEDADES TÉRMICAS DEL AISLANTE Y FLUIDO ---
k_madera = 0.12                 # Conductividad térmica del aislante de madera (W/m·K)
L_madera = 0.0254               # Espesor del aislante (1 pulgada = 0.0254 metros)
Cp_agua = 4186                  # Calor específico del agua (J/kg·K) - energía para elevar 1°C
SIGMA = 5.67e-8                 # Constante de Stefan-Boltzmann (W/m²·K⁴) - radiación térmica

# --- 1.6. GEOMETRÍA DEL SERPENTÍN (TUBO ESPIRAL) ---
# El serpentín es donde circula el agua a calentar
L_tubo_total = 15.0             # Longitud total de la espiral en metros
D_exterior = 0.5 * 0.0254       # Diámetro externo del tubo: 1/2" = 0.0127 metros
D_interno = (3/8) * 0.0254      # Diámetro interno del tubo: 3/8" = 0.00953 metros

# --- 1.7. CÁLCULO DE LA CAPACIDAD TOTAL DEL SERPENTÍN ---
# Volumen del agua en el serpentín y su masa correspondiente
v_total_m3 = (math.pi * D_interno**2 / 4) * L_tubo_total  # Volumen en m³ (fórmula cilindro)
masa_total_kg = v_total_m3 * 1000  # Masa en kg (densidad del agua ≈ 1000 kg/m³)


In [ ]:
# ============================================================================
# 2. APLICACIÓN DE OBSERVACIONES: VACIADO PARCIAL Y CÁLCULO DE ÁREA EFECTIVA
# ============================================================================
# Este bloque implementa optimizaciones basadas en observaciones del diseño real

# --- 2.1. VACIADO PARCIAL (Optimización de tiempo y eficiencia) ---
# En lugar de calentar toda la agua del serpentín, solo calentamos y vaciamos
# una fracción en cada ciclo. Esto permite optimizar el tiempo de calentamiento
# y aprovechar mejor el foco central de la lente concentradora.

factor_vaciado = 0.5            # Solo procesamos el 50% de la capacidad por ciclo
m_agua_batch = masa_total_kg * factor_vaciado  # Masa de agua por lote (kg)
v_litros_batch = m_agua_batch   # Volumen en litros (aproximación: 1 kg ≈ 1 litro de agua)

# --- 2.2. ÁREA DE PÉRDIDA EFECTIVA (Corrección por geometría de espiral) ---
# En sistemas convencionales, el tubo se analiza como una barra larga.
# Pero en nuestro diseño, el serpentín forma un espiral COMPACTO de 49 cm de diámetro.
# Las pérdidas térmicas no ocurren sobre los 15m desenrollados, sino sobre la 
# superficie efectiva del disco circular que forma la espiral.
# Por eso usamos el área de la lente como área de referencia para pérdidas.

area_perdida_efectiva = math.pi * (radio_lente**2)  # Área del círculo formado por la espiral

# --- 2.3. VISUALIZACIÓN DE PARÁMETROS CALCULADOS ---
# Imprimimos un resumen de los valores más importantes para verificación

print(f"--- Resumen de Geometría y Capacidad ---")
print(f"Capacidad total del tubo espiral: {masa_total_kg:.3f} kg (~ {masa_total_kg:.1f} L)")
print(f"Masa de agua por lote (Vaciado Parcial 50%): {m_agua_batch:.3f} kg")
print(f"Área de referencia para pérdidas (disco espiral): {area_perdida_efectiva:.4f} m²")
print(f"Relación de Concentración (Aa/A_pérdida): {Aa/area_perdida_efectiva:.2f}")
print()
print("→ Esta relación indica cuántas veces se concentra la energía solar")
print("→ Valores mayores = Mayor concentración = Mayor temperatura potencial")


--- Resumen de Geometría ---
Capacidad total del tubo: 1.069 Litros
Masa por lote (Vaciado Parcial): 0.534 kg
Área de pérdida corregida (Espiral compacto): 0.1886 m2
Relación de Concentración (Aa/A_perdida): 1.00


## 🔄 Paso 3: Estrategia de Vaciado Parcial

**Idea clave**: No calentar TODO el agua de una vez. En su lugar:

1. ✅ Calentar **solo 50%** de la capacidad = 4.5 litros
2. ✅ Esto tarda **2-4 horas** (en lugar de 8-16 horas)
3. ✅ Más ciclos = más flexibilidad
4. ✅ Mejor aprovechamiento del foco de la lente

**Resultado**: Producción de **8-15 L/día** (múltiples ciclos)

El área efectiva para pérdidas es el **disco circular** (0.188 m²), no los 15m desenrollados.


In [ ]:
# ============================================================================
# 3. GENERACIÓN DE DATOS SOLARES CON PVLIB
# ============================================================================
# PVLib es una librería estándar para cálculos solares que nos proporciona:
# - Posición del sol (elevación, azimut, ángulo cenital)
# - Radiación solar en diferentes geometrías
# - Componentes de radiación (directa, difusa, reflejada)

# --- 3.1. DEFINICIÓN DE LA UBICACIÓN ---
# Creamos un objeto Location con los parámetros de Morelia
morelia = pvlib.location.Location(
    latitude=phi,                  # Latitud (19.7008° N)
    longitude=longitud,            # Longitud (-101.1844° W)
    tz=zona_horaria,              # Zona horaria (para horas locales correctas)
    altitude=altitud_m,            # Altitud en metros (1920 m)
    name='Morelia'                # Identificador del lugar
)

# --- 3.2. GENERACIÓN DE SERIE DE TIEMPO ---
# Creamos un rango de fechas y horas para todo un mes (diciembre 2025)
# Frecuencia horaria: 60 minutos entre cada punto de dato
tiempos = pd.date_range(
    '2025-12-01 06:00:00',        # Fecha/hora inicial
    '2025-12-31 19:00:00',        # Fecha/hora final
    freq='60min',                 # Intervalo de 1 hora
    tz=zona_horaria               # Con información de zona horaria
)

# --- 3.3. CÁLCULO DE POSICIÓN SOLAR CON PVLIB ---
# Para cada instante, PVLib calcula ángulos solares usando algoritmos NREL estándar
df_solar_pd = morelia.get_solarposition(tiempos)
# Esto devuelve: aparente_elevation, elevation, azimuth, zenith, etc.


## 🌍 Paso 4: Obtener Datos del Sol con PVLib

**PVLib** es una librería estándar (NREL) que calcula:
- ☀️ **Posición del sol** cada hora (elevación, azimut, ángulo cenital)
- 📊 **Radiación solar** en diferentes geometrías
- 🕐 Maneja automáticamente las zonas horarias

**Salida**: Para todo diciembre 2025 (6:00 AM - 7:00 PM):
- ~400 registros horarios
- Fecha/hora, zenith, elevation, azimuth, etc.


In [ ]:
# ============================================================================
# 4. PREPARACIÓN DEL DATAFRAME DE PVLIB PARA POLARS
# ============================================================================
# PVLib devuelve un DataFrame de Pandas con el índice como fecha/hora.
# Necesitamos convertirlo a Polars (más eficiente) y prepararlo para uniones.

# --- 4.1. RESETEAR ÍNDICE Y RENOMBRAR COLUMNA ---
# Por defecto, la fecha/hora está en el índice. La convertimos a una columna llamada 'fecha_hora'
df_solar_pd = df_solar_pd.reset_index().rename(columns={'index': 'fecha_hora'})

# --- 4.2. ELIMINAR INFORMACIÓN DE ZONA HORARIA (Compatibilidad Polars) ---
# Polars tiene restricciones al convertir timestamps con zona horaria.
# Eliminamos la zona (pero los valores UTC permanecen correctos)
df_solar_pd['fecha_hora'] = df_solar_pd['fecha_hora'].dt.tz_localize(None)

# --- 4.3. CONVERSIÓN DE PANDAS A POLARS ---
# Polars es mucho más rápido para operaciones vectorizadas y uniones
# Este será nuestro dataframe principal de datos solares
df_pl = pl.from_pandas(df_solar_pd)
# Ahora df_pl contiene: fecha_hora, elevation, azimuth, zenith, etc.


## 🔄 Paso 5: Convertir de Pandas a Polars

PVLib devuelve datos en **Pandas**, pero nosotros usamos **Polars** (mucho más rápido).

**Lo que hacemos:**
1. El índice (fecha) → columna llamada `fecha_hora`
2. Eliminar zona horaria (Polars lo pide)
3. Convertir a Polars

**Resultado**: DataFrame de Polars listo para fusionar con datos NASA.


In [ ]:
# ============================================================================
# 5. CARGA Y PREPARACIÓN DE DATOS METEOROLÓGICOS (NASA)
# ============================================================================
# Los datos NASA proporcionan información real de temperatura ambiente y viento
# Este archivo contiene mediciones horarias de una estación meteorológica cercana a Morelia

# --- 5.1. CARGA DEL ARCHIVO CSV ---
# skip_rows=15 porque NASA incluye 15 líneas de encabezado/metadatos antes de los datos
df_nasa = pl.read_csv("Tabla datos NASA.csv", skip_rows=15)
cols = df_nasa.columns  # Obtenemos los nombres dinámicamente (por si cambian)

# --- 5.2. EXTRACCIÓN Y RENOMBRADO DE COLUMNAS RELEVANTES ---
# El archivo NASA tiene muchas columnas. Nos quedan solo 3: fecha/hora, temperatura, viento
# Usamos pl.datetime() para construir un timestamp a partir de columnas separadas (año, mes, día, hora)

df_nasa = df_nasa.select([
    # Construye un datetime a partir de columnas individuales (cols[0-3] = año, mes, día, hora)
    # dt.truncate("1h") fuerza minutos y segundos a 0 (solo horas exactas)
    # dt.cast_time_unit("us") convierte a microsegundos para precisión
    pl.datetime(pl.col(cols[0]), pl.col(cols[1]), pl.col(cols[2]), pl.col(cols[3]))
      .dt.truncate("1h")
      .dt.cast_time_unit("us")
      .alias("fecha_hora"),
    
    # Temperatura del aire en °C (conversión a Float64 para operaciones numéricas)
    pl.col(cols[7]).cast(pl.Float64).alias("temp_ambiente"),
    
    # Velocidad del viento en m/s (conversión a Float64)
    pl.col(cols[10]).cast(pl.Float64).alias("v_viento")
])
# Resultado: df_nasa con 3 columnas: fecha_hora, temp_ambiente, v_viento


## 🌡️ Paso 6: Cargar Datos Meteorológicos de NASA

**Archivo**: `Tabla datos NASA.csv`

Contiene datos horarios reales:
- 📊 **Temperatura ambiente** (°C)
- 💨 **Velocidad del viento** (m/s)
- Fechas de todo diciembre

**Proceso:**
1. Saltamos 15 líneas de encabezado
2. Extraemos solo 3 columnas: fecha/hora, temperatura, viento
3. Convertimos a formato de fecha compatible

**Resultado**: 3 columnas limpias listas para fusionar.


In [ ]:
# ============================================================================
# 6. PREPARACIÓN DE DATOS PARA UNIÓN
# ============================================================================
# Preparamos ambos dataframes (PVLib y NASA) para unirlos por fecha/hora

# --- 6.1. ALINEACIÓN TEMPORAL DE PVLIB ---
# Aseguramos que PVLib tenga exactamente horas exactas (para coincidir con NASA)
# truncate("1h") redondea a la hora más cercana
df_pl = df_pl.with_columns(
    pl.col("fecha_hora")
      .dt.truncate("1h")
      .dt.cast_time_unit("us")
)

# --- 6.2. DEFINICIÓN DE TEMPERATURAS DEL AGUA ---
# Estos son los puntos de operación del sistema: agua fría → caliente
Ti_c = 20.0                    # Temperatura inicial del agua (°C) - entrada
Tf_c = 94.0                    # Temperatura final del agua (°C) - salida (justo antes de hervir)
Tp_prom_c = (Ti_c + Tf_c) / 2  # Temperatura promedio del agua en el colector
Tp_k = Tp_prom_c + 273.15      # Temperatura promedio en Kelvin (para ecuaciones de radiación)

print(f"Rango de operación del agua:")
print(f"  Entrada: {Ti_c}°C | Salida: {Tf_c}°C | Promedio: {Tp_prom_c}°C")

# --- 6.3. UNIÓN DE TABLAS ---
# Combinamos datos de PVLib (posición solar) con datos NASA (clima local)
# La unión se realiza por la columna "fecha_hora" (left join, conservamos todas las horas de PVLib)
df_resultado = df_pl.join(df_nasa, on="fecha_hora")
# Resultado: cada fila tiene: fecha_hora, zenith, elevation, azimuth, temp_ambiente, v_viento


## 🔗 Paso 7: Preparar Unión de Datos

**Lo que hacemos:**
1. **Alinear horarios**: Ambos DataFrames a horas exactas (sin minutos)
2. **Definir rango de agua**: 20°C (entrada) → 94°C (salida)
3. **Fusionar**: PVLib + NASA en una sola tabla por fecha/hora

**Resultado**: Una tabla con TODO:
- ☀️ Datos del sol (posición, radiación)
- 🌡️ Clima (temperatura, viento)
- ⏰ Fecha/hora exacta

Lista para los cálculos.


In [ ]:
# ============================================================================
# 7. BLOQUE 1: DEFINICIÓN DE ECUACIONES (EXPRESIONES POLARS ESCALONADAS)
# ============================================================================
# Este bloque define TODAS las ecuaciones del modelo termosolar en forma de 
# expresiones Polars (lazy evaluation). Se aplicarán después en cascada.
# La ventaja: se optimizan automáticamente y se ejecutan en memoria de forma vectorizada.

# NOTA IMPORTANTE: Las expresiones se organizan en 8 ESCALONES para evitar
# dependencias circulares. Cada escalón puede usar resultados del anterior.

# --- ESCALÓN 1: VARIABLES INDEPENDIENTES Y AMBIENTALES ---

# Constantes del colector (verificación)
altura_techo = 3.0              # Altura del colector desde el suelo (metros)
alpha_terreno = 0.22            # Coeficiente de fricción para zona suburbana (0.22 típico para casas/árboles)

# ✓ Variable lógica: ¿Es de día? (zenith < 90° significa el sol está sobre el horizonte)
expr_es_de_dia = (pl.col("zenith") < 90).alias("es_de_dia")

# ✓ Variable lógica: ¿Está en ventana productiva? (10:00 - 18:00, período de mayor producción)
expr_ventana_prod = pl.col("fecha_hora").dt.hour().is_between(10, 18, closed="left").alias("ventana_productiva")

# ✓ Temperatura del aire ambiente en Kelvin (conversión de °C para radiación térmica)
expr_ta_k = (pl.col("temp_ambiente") + 273.15).alias("Ta_k")

# ✓ Energía requerida para calentar un lote de agua desde Ti a Tf
# Fórmula: E = m * Cp * ΔT donde ΔT = Tf - Ti
expr_e_lote = pl.lit(m_agua_batch * Cp_agua * (Tf_c - Ti_c)).alias("E_lote_joules")

# ✓ Radiación solar extraterrestre (varía con la estación del año)
# Fórmula de Spencer con corrección del día del año: Gon = Gsc * [1 + 0.033*cos(360*n/365)]
expr_gon = (Gsc * (1 + 0.033 * ((360 * pl.col("fecha_hora").dt.ordinal_day() / 365) * math.pi / 180).cos())).alias("Gon")

# --- ESCALÓN 2: ÁNGULOS SOLARES Y TEMPERATURA DEL CIELO ---

# ✓ Coseno del ángulo cenital (solo durante el día, =1.0 por la noche)
# zenith es el ángulo entre el sol y la vertical, cos(zenith) es importante para la radiación
expr_cos_theta_z = pl.when(pl.col("es_de_dia")).then(
    (pl.col("zenith") * math.pi / 180).cos()
).otherwise(1.0).alias("cos_theta_z")

# ✓ Temperatura efectiva del cielo (modela radiación térmica del cielo nocturno)
# Fórmula empírica: T_sky ≈ 0.0552 * Ta^1.5 (Swinbank)
expr_t_sky = (0.0552 * (pl.col("Ta_k")**1.5)).alias("T_sky")

# --- ESCALÓN 3: VIENTO Y TRANSFERENCIA CONVECTIVA ---

# ✓ Velocidad del viento real a la altura del colector (ley logarítmica de perfil de viento)
# En superficies rugosas, el viento aumenta con la altura: V(z) = V(10m) * (z/10)^α
expr_viento_real = (pl.col("v_viento") * (altura_techo / 10.0)**alpha_terreno).alias("v_viento_techo")

# ✓ Coeficiente de transferencia convectiva de McAdams (para aire forzado)
# hw = 5.7 + 3.8 * V  donde V es la velocidad del viento en m/s
expr_hw = (5.7 + 3.8 * pl.col("v_viento_techo")).alias("hw")

# --- ESCALÓN 4: RADIACIÓN SOLAR DIRECTA ---

# ✓ Radiación solar normal (perpendicular a los rayos solares)
# Fórmula de Hottel: Gb_n = Gon * [Ao + A1 * exp(-K/cos_zenith)]
# A0, A1, K son coeficientes que varían con la altitud
expr_gb_n = (pl.col("Gon") * (Ao + A1 * (-K / pl.col("cos_theta_z")).exp())).alias("gb_n")

# ✓ Coeficiente de transferencia radiativa (radiación térmica del tubo caliente al cielo frío)
# Fórmula de Stefan-Boltzmann linealizada: h_rad = ε * σ * (Tp + T_sky) * (Tp² + T_sky²)
expr_h_rad = (epsilon_p * SIGMA * (Tp_k + pl.col("T_sky")) * (Tp_k**2 + pl.col("T_sky")**2)).alias("h_rad")

# --- ESCALÓN 5: COEFICIENTES DE PÉRDIDA TÉRMICA (U_L) ---
# U_L = hw + h_rad + (k/L) → pérdida de calor por unidad de área y ΔT

# ✓ Escenario 1: Espiral DESNUDA (sin protección) - sufre todo el viento y radiación
expr_u_l_desnudo = (
    (pl.col("hw") + pl.col("h_rad")) + (k_madera / L_madera)
).alias("U_L_Desnudo")

# ✓ Escenario 2: Espiral en CAJA CON VIDRIO (protegida) - U_L fijo empírico
# El vidrio reduce convección (bloquea viento) y transmite 90% de radiación
expr_u_l_caja = (pl.lit(6.5) + (k_madera / L_madera)).alias("U_L_Caja")

# --- ESCALÓN 6: CALOR SOLAR ÚTIL ABSORBIDO (Q_opt) ---

# ✓ Escenario 1: Calor absorbido en tubo DESNUDO
# Q_opt = Gb_n * Aa * τ_lente * α_cobre
expr_q_opt_desnudo = (pl.col("gb_n") * Aa * tau_lente * alpha_cobre).alias("Q_opt_Desnudo")

# ✓ Escenario 2: Calor absorbido en tubo CON VIDRIO
# Q_opt = Gb_n * Aa * τ_lente * τ_vidrio * α_cobre (el vidrio atenúa un 10% adicional)
expr_q_opt_caja = (pl.col("gb_n") * Aa * tau_lente * tau_vidrio * alpha_cobre).alias("Q_opt_Caja")

# --- ESCALÓN 7: PÉRDIDAS TÉRMICAS (Q_loss) ---

# ✓ Escenario 1: Pérdidas en tubo DESNUDO
# Q_loss = U_L * A_pérdida * ΔT donde ΔT = Tp - Ta
expr_q_loss_desnudo = (
    pl.col("U_L_Desnudo") * area_perdida_efectiva * (Tp_prom_c - pl.col("temp_ambiente"))
).alias("Q_loss_Desnudo")

# ✓ Escenario 2: Pérdidas en tubo CON VIDRIO
# Q_loss = U_L_Caja * A_pérdida * ΔT (menor U_L → menores pérdidas)
expr_q_loss_caja = (
    pl.col("U_L_Caja") * area_perdida_efectiva * (Tp_prom_c - pl.col("temp_ambiente"))
).alias("Q_loss_Caja")

# --- ESCALÓN 8: BALANCE TÉRM ICO NETO (CALOR ÚTIL) ---

# ✓ Escenario 1: Calor disponible para calentar agua (DESNUDO)
# Qu = Q_opt - Q_loss (lo que absorbemos menos lo que perdemos)
expr_qu_desnudo = (pl.col("Q_opt_Desnudo") - pl.col("Q_loss_Desnudo")).alias("Qu_Desnudo_W")

# ✓ Escenario 2: Calor disponible para calentar agua (CON VIDRIO - RECOMENDADO)
expr_qu_caja = (pl.col("Q_opt_Caja") - pl.col("Q_loss_Caja")).alias("Qu_Caja_W")

# --- ESCALÓN 9: TIEMPO PARA HERVIR UN LOTE ---

# ✓ Tiempo requerido para calentar un lote de agua desde Ti a Tf
# Fórmula: t = E / Qu   donde E es la energía en Joules y Qu es la potencia en Watts
# Solo calculamos si: hay sol (ventana productiva) y hay calor útil positivo
expr_tiempo_herv = pl.when(
    pl.col("ventana_productiva") & (pl.col("Qu_Caja_W") > 0)
).then(
    (pl.col("E_lote_joules") / pl.col("Qu_Caja_W")) / 60  # Dividimos por 60 para convertir seg a min
).otherwise(None).alias("Tiempo_Hervir_min")

# --- ESCALÓN 10: PRODUCCIÓN EN LITROS POR HORA ---

# ✓ Producción horaria: si conseguimos hervir un lote en t minutos, 
# entonces en 60 minutos (1 hora) hervimos 60/t lotes
expr_litros_hora = pl.when(
    pl.col("Tiempo_Hervir_min") > 0
).then(
    (60 / pl.col("Tiempo_Hervir_min")) * v_litros_batch
).otherwise(0.0).alias("Litros_Hora")

# --- ESCALÓN 11: EFICIENCIA TÉRMICA ---

# ✓ Eficiencia = (Calor útil / Radiación incidente) * 100
# Indica qué porcentaje de la energía solar se convierte en calor del agua
expr_eficiencia = pl.when(
    pl.col("gb_n") > 0
).then(
    (pl.col("Qu_Caja_W") / (pl.col("gb_n") * Aa)) * 100
).otherwise(0.0).alias("Eficiencia_pct")

# ✓ Irradiancia horizontal global (GHI): suma directa + difusa en plano horizontal
expr_ghi = (pl.col("gb_n") * pl.col("cos_theta_z")).alias("GHI")

# ✓ Temperatura final del agua (asumimos que queremos agua a 94°C)
expr_t_final = pl.lit(94.0).alias("T_agua_final_C")


## ⚙️ Paso 8: BLOQUE 1 - Definir 11 Escalones de Ecuaciones

Este es el **corazón del proyecto**. Definimos TODAS las ecuaciones en orden:

| Escalón | Qué calcula | Ejemplo |
|---------|------------|---------|
| 1️⃣ | Variables base (es de día, viento real, etc.) | ¿Es mediodía? |
| 2️⃣ | Ángulos solares (cos zenith, T cielo) | ¿Qué ángulo tiene el sol? |
| 3️⃣ | Viento ajustado por altura | ¿Cuánto viento en el colector? |
| 4️⃣ | Radiación solar directa | ¿Cuánta energía cae? |
| 5️⃣ | Coeficientes de pérdida (U_L) | ¿Cuánto calor se pierde? |
| 6️⃣ | Calor absorbido (Q_opt) | ¿Cuánto gano? |
| 7️⃣ | Pérdidas térmicas (Q_loss) | ¿Cuánto pierdo? |
| 8️⃣ | Calor útil neto (Qu) | ¿Cuánto queda? ⭐ |
| 9️⃣ | Tiempo para hervir | ¿Cuántos minutos? |
| 🔟 | Litros/hora | ¿Cuánta agua/hora? |
| 1️⃣1️⃣ | Eficiencia | ¿Qué % de energía aprovecho? |

**Clave**: El orden importa. Cada escalón usa resultados del anterior.


In [ ]:
# ============================================================================
# 8. BLOQUE 2: APLICACIÓN EN CASCADA (ESCALONADO) DE TODAS LAS ECUACIONES
# ============================================================================
# Aquí aplicamos todas las expresiones definidas en el Bloque 1.
# El orden de los escalones es CRÍTICO: cada uno depende de los anteriores.
# Polars optimiza automáticamente todas las operaciones en una sola pasada.

df_calculado = (
    # Punto de partida: unión de datos PVLib + NASA
    df_pl.join(df_nasa, on="fecha_hora")
    
    # ESCALÓN 1: Variables independientes
    # Calcula: es_de_dia, ventana_productiva, Ta_k, v_viento_techo, E_lote_joules, Gon
    .with_columns([expr_es_de_dia, expr_ventana_prod, expr_ta_k, expr_viento_real, expr_e_lote, expr_gon])
    
    # ESCALÓN 2: Ángulos solares y temperatura del cielo
    # Usa: es_de_dia (escalón 1) | Calcula: cos_theta_z, T_sky, hw
    .with_columns([expr_cos_theta_z, expr_t_sky, expr_hw])
    
    # ESCALÓN 3: Radiación solar
    # Usa: Gon, cos_theta_z (escalón 1-2) | Calcula: gb_n, h_rad
    .with_columns([expr_gb_n, expr_h_rad])
    
    # ESCALÓN 4: Coeficientes de pérdida térmica (U_L)
    # Usa: hw, h_rad (escalón 2-3) | Calcula: U_L_Desnudo, U_L_Caja
    .with_columns([expr_u_l_desnudo, expr_u_l_caja])
    
    # ESCALÓN 5: Calor solar absorbido (Q_opt)
    # Usa: gb_n (escalón 3) | Calcula: Q_opt_Desnudo, Q_opt_Caja
    .with_columns([expr_q_opt_desnudo, expr_q_opt_caja])
    
    # ESCALÓN 6: Pérdidas térmicas (Q_loss)
    # Usa: U_L_Desnudo, U_L_Caja (escalón 4) | Calcula: Q_loss_Desnudo, Q_loss_Caja
    .with_columns([expr_q_loss_desnudo, expr_q_loss_caja])
    
    # ESCALÓN 7: Balance térmico neto (Qu)
    # Usa: Q_opt, Q_loss (escalón 5-6) | Calcula: Qu_Desnudo_W, Qu_Caja_W
    .with_columns([expr_qu_desnudo, expr_qu_caja])
    
    # ESCALÓN 8: Tiempo de hervor
    # Usa: ventana_productiva, E_lote_joules, Qu_Caja_W (escalón 1, 7)
    # Calcula: Tiempo_Hervir_min
    .with_columns([expr_tiempo_herv])
    
    # ESCALÓN 9: Métricas finales de producción y eficiencia
    # Usa: Tiempo_Hervir_min, gb_n (escalón 8, 3)
    # Calcula: Litros_Hora, Eficiencia_pct, GHI, T_agua_final_C
    .with_columns([expr_litros_hora, expr_eficiencia, expr_ghi, expr_t_final])
)

print("✓ Cálculos completados exitosamente")
print(f"✓ Total de filas procesadas: {df_calculado.height}")
print(f"✓ Total de columnas: {df_calculado.width}")


## 🔄 Paso 9: BLOQUE 2 - Aplicar Ecuaciones en Cascada

Ahora ejecutamos los 11 escalones **en orden**, para cada hora del mes.

**Polars optimiza automáticamente** todo esto en una sola pasada eficiente.

```
Entrada: Tabla con sol + clima (400 filas)
     ↓
 Escalón 1 → Escalón 2 → ... → Escalón 11
     ↓
Salida: Tabla con TODOS los cálculos (400 filas completadas)
```

**Resultado**: Para cada hora tenemos:
- ☀️ Radiación solar
- 💨 Viento real
- 🔥 Calor útil (Qu)
- ⏱️ Tiempo para hervir
- 📊 Producción (L/hora)


In [ ]:
# ============================================================================
# 9. BLOQUE 3: ACUMULADOS, RENOMBRADO Y FILTRADO FINAL
# ============================================================================
# En este bloque procesamos los resultados finales:
# - Calculamos acumulados por día
# - Renombramos columnas para mayor claridad
# - Filtramos solo la ventana productiva (10:00 - 18:00)
# - Preparamos tabla final para visualización

# --- 9.1. CÁLCULO DE ACUMULADOS DIARIOS ---
df_resultados = (
    df_calculado
    
    # Calcula litros acumulados por cada día (usa cum_sum sobre particiones diarias)
    .with_columns([
        pl.col("Litros_Hora")           # Toma la columna de litros por hora
          .fill_null(0)                 # Reemplaza None con 0 (noches, días nublados)
          .cum_sum()                    # Suma acumulativa
          .over(                        # ...pero reseteada para cada día
              pl.col("fecha_hora").dt.date()  # Agrupa por fecha (año-mes-día)
          )
          .alias("Litros_Acum_Dia")    # Rename a "Litros acumulados del día"
    ])
    
    # Renombramos la columna de radiación para mayor claridad
    .rename({"gb_n": "G_Directa_Normal"})
)

# --- 9.2. SELECCIÓN DE COLUMNAS CLAVE PARA LA TABLA FINAL ---
# Solo mostramos el "horario productivo" (10:00 - 18:00) y las columnas más importantes
# Esto hace la tabla mucho más legible que mostrar todos los cálculos intermedios

df_tabla_final = (
    df_resultados
    .filter(pl.col("ventana_productiva"))  # Solo horas 10:00 - 18:00
    .select([
        "fecha_hora",                 # Fecha y hora
        "G_Directa_Normal",           # Irradiancia solar directa normal (W/m²)
        "v_viento_techo",             # Velocidad del viento ajustada a la altura del colector
        "v_viento",                   # Velocidad del viento a 10 m de altura (referencia)
        "Qu_Desnudo_W",               # Calor útil del diseño DESNUDO (W) - solo para comparación
        "Qu_Caja_W",                  # Calor útil del diseño CON CAJA Y VIDRIO (W) - RECOMENDADO ⭐
        "Tiempo_Hervir_min",          # Tiempo para hervir 1 lote de agua (minutos)
        "Eficiencia_pct",             # Eficiencia térmica del colector (%)
        "Litros_Hora",                # Producción de agua caliente por hora (litros)
        "Litros_Acum_Dia"             # Agua caliente producida acumulada del día (litros)
    ])
)

# --- 9.3. IMPRESIÓN DE RESULTADOS Y TABLA FINAL ---
print("=" * 80)
print("PRODUCCIÓN ESTIMADA DE AGUA CALIENTE")
print("Período: Ventana productiva (10:00 - 18:00)")
print("Diseño: Espiral en CAJA CON VIDRIO (protegida del viento)")
print("=" * 80)
print()

# Mostrar las primeras 12 filas (representa aproximadamente un día de producción)
print(f"Primeros 12 registros (horarios desde 10:00):")
print(df_tabla_final[0:12])


## 📊 Paso 10: BLOQUE 3 - Mostrar Resultados Finales

**Lo que hacemos:**
1. ✅ Calcular **litros acumulados por día** (suma total)
2. ✅ Filtrar solo **horas productivas** (10:00 - 18:00)
3. ✅ Seleccionar **10 columnas clave** (descartar intermedios)
4. ✅ Mostrar tabla legible

**Columnas importantes:**
| Columna | Significado |
|---------|-------------|
| `fecha_hora` | Hora del día |
| `G_Directa_Normal` | Radiación solar (W/m²) |
| `Qu_Caja_W` | **Calor útil** - lo importante |
| `Tiempo_Hervir_min` | ¿Cuánto tarda 1 lote? |
| `Litros_Hora` | **Producción real** |
| `Litros_Acum_Dia` | **Total acumulado** |

🎯 **Ahora ya tienes la respuesta final**: ¿Cuántos litros de agua caliente puedes producir?
